In [1]:
from youtube_transcript_api import  YouTubeTranscriptApi, TranscriptsDisabled
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_classic.vectorstores import FAISS
load_dotenv()

True

In [23]:
video_id = "Gfr50f6ZBvo" # only the ID, not full URL
try:
    # If you don’t care which language, this returns the “best” one
    transcript_list = YouTubeTranscriptApi().fetch(video_id)

    # Flatten it to plain text
    transcript = ' '.join(f'"{f"{snippet.text},start: {snippet.start},duration: {snippet.duration}"}"' for snippet in transcript_list if snippet.text is not None)
    print(transcript)
#
except TranscriptsDisabled:
    print("No captions available for this video.")

"the following is a conversation with,start: 0.08,duration: 3.44" "demus hasabis,start: 1.76,duration: 4.96" "ceo and co-founder of deepmind,start: 3.52,duration: 5.119" "a company that has published and builds,start: 6.72,duration: 4.48" "some of the most incredible artificial,start: 8.639,duration: 4.561" "intelligence systems in the history of,start: 11.2,duration: 4.8" "computing including alfred zero that,start: 13.2,duration: 3.68" "learned,start: 16.0,duration: 2.96" "all by itself to play the game of gold,start: 16.88,duration: 4.559" "better than any human in the world and,start: 18.96,duration: 5.6" "alpha fold two that solved protein,start: 21.439,duration: 4.241" "folding,start: 24.56,duration: 4.16" "both tasks considered nearly impossible,start: 25.68,duration: 5.439" "for a very long time,start: 28.72,duration: 4.4" "demus is widely considered to be one of,start: 31.119,duration: 4.481" "the most brilliant and impactful humans,start: 33.12,duration: 4.0" "in the history 

In [24]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [25]:
chunks[100]

Document(metadata={}, page_content='"google search,start: 2511.76,duration: 4.319" "and so there\'s a data set on which it\'s,start: 2513.119,duration: 5.281" "trained and how to map this amino acids,start: 2516.079,duration: 3.52" "because first of all it\'s incredible,start: 2518.4,duration: 3.04" "that a protein this little chemical,start: 2519.599,duration: 3.52" "computer is able to do that computation,start: 2521.44,duration: 4.159" "itself in some kind of distributed way,start: 2523.119,duration: 4.561" "and do it very quickly,start: 2525.599,duration: 3.841" "that\'s a weird thing and they evolved,start: 2527.68,duration: 2.8" "that way because you know in the,start: 2529.44,duration: 2.08" "beginning,start: 2530.48,duration: 2.879" "i mean that\'s a great invention just the,start: 2531.52,duration: 4.4" "protein itself yes i mean and then they,start: 2533.359,duration: 5.441" "there\'s i think probably a history of,start: 2535.92,duration: 5.12" "like uh they evolved,start: 25

In [26]:
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")
vector_store = FAISS.from_documents(chunks, embeddings)

GoogleGenerativeAIError: Error embedding content (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/embed_content_free_tier_requests, limit: 100, model: gemini-embedding-2, description: Embed content requests per minute limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.\nPlease retry in 13.343134101s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://aistudio.google.com/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': 'Embed content requests per minut

In [27]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

NameError: name 'vector_store' is not defined

In [28]:
retriever.invoke('What is deepmind')

NameError: name 'retriever' is not defined

In [29]:
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0.2)

In [30]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [31]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

NameError: name 'retriever' is not defined

In [32]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

NameError: name 'retrieved_docs' is not defined

In [33]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

NameError: name 'context_text' is not defined

In [34]:
answer = llm.invoke(final_prompt)
print(answer.content)

NameError: name 'final_prompt' is not defined

In [35]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [36]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [37]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

NameError: name 'retriever' is not defined

In [38]:
parallel_chain.invoke('who is Demis')

NameError: name 'parallel_chain' is not defined

In [39]:
parser = StrOutputParser()

In [40]:
main_chain = parallel_chain | prompt | llm | parser

NameError: name 'parallel_chain' is not defined

In [41]:
main_chain.invoke('Can you summarize the video')

NameError: name 'main_chain' is not defined